# Bank GoodCredit — Credit Risk Prediction

# Step 1 — Project Introduction

### Definition
This section defines the business problem, target variable, and overall objective of the credit-risk project.

### Why this step is important
Before performing data analysis or machine learning, we need to clearly understand what the bank wants to predict and why the prediction is useful.

### Business Problem
Bank GoodCredit wants to predict the credit risk of its current credit-card customers.

The objective is to identify customers who are more likely to have bad credit behavior and thereby help reduce credit-default risk.

### Target Variable
The target variable is `Bad_label`.

- `Bad_label = 0` → Customer has good credit history.
- `Bad_label = 1` → Customer has bad credit history / falls into the 30+ DPD bucket.

### Project Objectives
1. Understand the available customer data.
2. Perform data cleaning and exploratory data analysis.
3. Create meaningful customer-level features.
4. Build machine-learning models.
5. Evaluate model performance using Gini.
6. Perform decile/rank-ordering analysis.
7. Identify important features.
8. Compare the final model with the benchmark Gini of 0.379 (37.9%).

### Expected Output
A clear understanding of the business problem, target variable, and modelling objectives.

### Insight
The project is fundamentally a **credit-risk classification problem**: the model will learn from customer information and predict whether the customer belongs to the good-credit or bad-credit class.

**Source note:** The business case states that Bank GoodCredit wants to predict credit score/creditworthiness for current credit-card customers, with `Bad_label` as the target and 30+ DPD defining the bad-credit class.

# Step 2 — Import Required Libraries

### Definition
This step imports the Python libraries required for data manipulation, visualization, database connectivity, and later analysis.

### Why this step is important
The project involves large tabular datasets, database access, exploratory analysis, feature engineering, and machine learning, so we need the appropriate Python tools available before starting.

### Libraries Used
- `pandas` → Data manipulation and analysis.
- `numpy` → Numerical calculations.
- `matplotlib` → Data visualization.
- `seaborn` → Statistical visualization.
- `sqlalchemy` → Database connection.
- `getpass` → Secure password entry.
- `warnings` → Control unnecessary warning messages.

### Expected Output
`All required libraries imported successfully.`

### Insight
The Python environment is now prepared for the data-analysis workflow.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Database connection
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from getpass import getpass

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("All required libraries imported successfully.")

# Step 3 — Connect to MySQL Database

### Definition
A database connection establishes communication between Python and the MySQL database that stores the Bank GoodCredit project data.

### Why this step is important
The project data is stored in MySQL, so we must establish a working connection before retrieving the source tables.

### Database Details
- Database type → MySQL
- Database name → `project_banking`
- Port → `3306`
- Username → `YOUR_MYSQL_USERNAME`
- Host → Provided project MySQL server

### Security
The password is entered using `getpass()` so it is not displayed or hard-coded in the notebook.

### Important Technical Point
We use `SQLAlchemy URL.create()` instead of manually constructing a connection string, because the project password contains special characters and should not be interpreted as part of the host/URL.

### Expected Output
`MySQL connection successful.`

### Insight
A successful connection confirms that Python can communicate with the project's database and that we can proceed to load the three source tables.

In [ ]:
# Database details
username = "YOUR_MYSQL_USERNAME"
password = getpass("Enter MySQL password: ")

host = "YOUR_MYSQL_HOST"
port = 3306
database = "project_banking"

# Create the connection safely.
# URL.create() keeps the password separate from the host,
# so special characters in the password are handled correctly.
connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database
)

# Create SQLAlchemy engine
engine = create_engine(connection_url)

# Test the connection
with engine.connect() as connection:
    print("MySQL connection successful.")

# Step 4 — Load the Three Project Tables

### Definition
This step retrieves the three source tables from MySQL and stores them as Pandas DataFrames.

### Why this step is important
Each table contains a different type of customer information, and these sources will later be transformed into a customer-level feature matrix.

### Source Tables
1. `Cust_Account` → Historical account and payment information.
2. `Cust_Enquiry` → Historical credit-enquiry information such as enquiry amount and purpose.
3. `Cust_Demographics` → Current customer/application demographic information and the target variable.

### Expected Output
Three Pandas DataFrames:
- `acct`
- `enq`
- `demo`

### Insight
The three tables contain complementary information, so we will not simply concatenate them row-by-row. Account and enquiry records can occur multiple times for a customer; later we will aggregate them to customer level before modelling.

In [ ]:
# Load Customer Account data
acct = pd.read_sql(
    "SELECT * FROM Cust_Account",
    engine
)

# Load Customer Enquiry data
enq = pd.read_sql(
    "SELECT * FROM Cust_Enquiry",
    engine
)

# Load Customer Demographics data
demo = pd.read_sql(
    "SELECT * FROM Cust_Demographics",
    engine
)

print("All three tables loaded successfully.")

# Step 5 — Check Dataset Size

### Definition
The Pandas `shape` attribute returns the number of rows and columns in a dataset.

For example, `(1000, 20)` means:
- 1,000 rows/records.
- 20 columns/variables.

### Why this step is important
Dataset size tells us the scale of the project and helps us understand the relationship between transaction/history tables and the customer-level table.

### Expected Output
The current database output shown in the working notebook is approximately:

- Account Data → `(186329, 21)`
- Enquiry Data → `(413188, 6)`
- Demographics Data → `(23896, 83)`

### Insight
The tables have very different numbers of records.

- `Cust_Account` has many account-level records.
- `Cust_Enquiry` has many enquiry-level records.
- `Cust_Demographics` is much smaller and represents the customer/application-level dataset.

Therefore, we should **not perform a direct row-wise merge** of all three tables. Account and enquiry information will need to be aggregated at customer level and then joined to the customer-level data.

In [ ]:
# Check the number of rows and columns
print("Account Data      :", acct.shape)
print("Enquiry Data      :", enq.shape)
print("Demographics Data :", demo.shape)

# Step 6 — Understand Data Structure

### Definition
Data structure analysis means checking the sample records, column names, and data types of each dataset.

### Why this step is important
Before cleaning or modelling, we need to understand what information is available and how each variable is stored.

### Objectives
1. View sample records.
2. Identify column names.
3. Identify data types.
4. Understand the structure of each table.

### Expected Output
We should see the first few rows and the data types for the Account, Enquiry, and Demographics tables.

### Insight
We are looking for date fields, numeric variables, categorical variables, the customer identifier, payment-history fields, enquiry fields, and the target variable `Bad_label`.

In [ ]:
# Display first 5 records from each dataset

print("ACCOUNT DATA")
display(acct.head())

print("\nENQUIRY DATA")
display(enq.head())

print("\nDEMOGRAPHICS DATA")
display(demo.head())

In [ ]:
# Check data types

print("ACCOUNT DATA TYPES")
print(acct.dtypes)

print("\nENQUIRY DATA TYPES")
print(enq.dtypes)

print("\nDEMOGRAPHICS DATA TYPES")
print(demo.dtypes)

# Step 7 — Missing Value Analysis

### Definition
Missing-value analysis identifies variables containing null or unavailable observations.

### Why this step is important
Missing values can affect calculations, feature engineering, statistical analysis, and machine-learning models.

### Objective
Calculate the number and percentage of missing values in every column.

### Expected Output
A table showing:
- `Missing_Count`
- `Missing_Percentage`

sorted from the highest missing percentage to the lowest.

### Insight
We should **not immediately fill or remove missing values**. First we need to understand which variables have missing values and how much information is missing; the treatment will depend on the actual result.

In [ ]:
def missing_summary(df):
    summary = pd.DataFrame({
        "Missing_Count": df.isnull().sum(),
        "Missing_Percentage": df.isnull().mean() * 100
    })

    return summary.sort_values(
        "Missing_Percentage",
        ascending=False
    )

print("ACCOUNT DATA - Missing Values")
display(missing_summary(acct))

print("ENQUIRY DATA - Missing Values")
display(missing_summary(enq))

print("DEMOGRAPHICS DATA - Missing Values")
display(missing_summary(demo))

# Step 8 — Duplicate Record Analysis

### Definition
Duplicate analysis identifies records that appear more than once with identical values across all columns.

### Why this step is important
Exact duplicate records can artificially increase the frequency of observations and may distort descriptive analysis, feature creation, and model training.

### Objective
Count completely duplicated rows in each dataset.

### Expected Output
The number of exact duplicate records for Account, Enquiry, and Demographics data.

### Insight
A repeated `customer_no` does **not** automatically mean a duplicate. One customer can legitimately have multiple accounts or multiple credit enquiries, so we will only investigate exact duplicates at this stage.

In [ ]:
print("Duplicate records in Account Data      :", acct.duplicated().sum())
print("Duplicate records in Enquiry Data      :", enq.duplicated().sum())
print("Duplicate records in Demographics Data :", demo.duplicated().sum())

# Step 9 — Identify Customer Key and Target Variable

### Definition
The customer key identifies the customer whose information is distributed across the project tables.

The target variable is the outcome that the machine-learning model will learn to predict.

### Customer Key
The project documentation uses `customer_no` in the Account and Enquiry tables and identifies customer number in the Demographics table.

### Target Variable
The target variable is `Bad_label`.

- `0` → Customer has good credit history.
- `1` → Customer has bad credit history / falls into the 30+ DPD bucket.

### Why this step is important
We need the correct customer identifier before aggregating Account and Enquiry information. We also need to verify the target distribution before modelling.

### Objective
Confirm the customer identifier and inspect the target distribution.

### Expected Output
The output should confirm whether `customer_no` and `Bad_label` are available and show the number and percentage of observations in each target class.

### Insight
If one target class is much smaller than the other, the data is class-imbalanced. That will influence model evaluation and later modelling decisions.

In [ ]:
# Check customer identifier
print("Customer ID in Account       :", "customer_no" in acct.columns)
print("Customer ID in Enquiry       :", "customer_no" in enq.columns)
print("Customer ID in Demographics  :", "customer_no" in demo.columns)

# Check target variable
print("\nTarget column available      :", "Bad_label" in demo.columns)

# Target distribution
print("\nBad_label distribution:")
display(demo["Bad_label"].value_counts(dropna=False))

print("\nBad_label percentage:")
display(
    demo["Bad_label"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
)

# Step 9 — Data Relationship Diagram

The following conceptual diagram shows how the three source tables will ultimately contribute to the modelling dataset.

In [ ]:
# Conceptual data-flow diagram
# This is a visualization only; the actual joins/aggregations will be
# performed after we inspect the data in the next stages.

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")

boxes = [
    (0.05, 0.55, 0.24, 0.25, "Cust_Account\nAccount + Payment History"),
    (0.38, 0.55, 0.24, 0.25, "Customer-Level\nFeature Matrix"),
    (0.71, 0.55, 0.24, 0.25, "ML Model\nPredict Bad_label"),
    (0.05, 0.12, 0.24, 0.25, "Cust_Enquiry\nCredit Enquiry History"),
    (0.71, 0.12, 0.24, 0.25, "Cust_Demographics\nCustomer + Target")
]

for x, y, w, h, label in boxes:
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, linewidth=1.5))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(0.38, 0.675), xytext=(0.29, 0.675),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.71, 0.675), xytext=(0.62, 0.675),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.38, 0.61), xytext=(0.29, 0.28),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.62, 0.61), xytext=(0.71, 0.28),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))

ax.set_title("Bank GoodCredit — Conceptual Data Flow", fontsize=14)
plt.show()

### Diagram Insight
The diagram represents the intended project flow: Account and Enquiry histories will be transformed into customer-level features, combined with the customer-level data, and then used to predict `Bad_label`. The exact aggregation and join logic will be based on the actual columns and data quality observed in the notebook.

# Step 10 — Target Variable Visualization

### Definition
Target visualization shows the distribution of good-credit and bad-credit customers using a chart.

### Why this step is important
A visual view makes the target-class distribution easier to interpret and helps us identify potential class imbalance.

### Objective
Visualize the distribution of `Bad_label` and calculate the exact class percentages.

### Expected Output
A count plot with:
- `0` → Good credit history.
- `1` → Bad credit history / 30+ DPD.

A second output will show the exact percentage for each class.

### Insight
If `Bad_label = 1` is substantially smaller than `Bad_label = 0`, the target is imbalanced. This is important because a model can appear accurate simply by predicting the majority class, so later evaluation must use appropriate risk-model metrics such as Gini and rank ordering.

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=demo,
    x="Bad_label"
)

plt.title("Distribution of Bad_label")
plt.xlabel("Bad Label")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
target_percentage = (
    demo["Bad_label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("Target Distribution (%)")
display(target_percentage)

# Step 11 — Customer-Level Uniqueness Analysis

### Definition
Customer-level uniqueness analysis checks how many unique customers are present in each source table and how frequently each customer appears.

### Why this step is important
The Account and Enquiry tables can contain multiple records for the same customer, while the Demographics table is closer to the customer/application level.

We need to confirm this structure before aggregating data.

### Objective
1. Count unique customers.
2. Count total records.
3. Calculate the average number of records per customer.
4. Identify customers with multiple records.

### Expected Output
A summary for Account, Enquiry, and Demographics showing total records and unique customers.

### Insight
If the number of records is much larger than the number of unique customers, that table contains multiple records per customer and must be aggregated before creating the final customer-level feature matrix.

In [ ]:
# Customer-level summary

datasets = {
    "Account": acct,
    "Enquiry": enq,
    "Demographics": demo
}

customer_summary = []

for name, df in datasets.items():
    if "customer_no" in df.columns:
        total_records = len(df)
        unique_customers = df["customer_no"].nunique()
        avg_records = total_records / unique_customers if unique_customers else np.nan

        customer_summary.append({
            "Dataset": name,
            "Total_Records": total_records,
            "Unique_Customers": unique_customers,
            "Avg_Records_Per_Customer": avg_records
        })

customer_summary = pd.DataFrame(customer_summary)

display(customer_summary)

In [ ]:
# Frequency distribution of records per customer

for name, df in [("Account", acct), ("Enquiry", enq), ("Demographics", demo)]:
    if "customer_no" in df.columns:
        frequency = df.groupby("customer_no").size()

        print(f"\n{name} Data")
        print("Customers with exactly 1 record :", (frequency == 1).sum())
        print("Customers with more than 1 record:", (frequency > 1).sum())
        print("Maximum records for one customer:", frequency.max())

# Step 12 — Identify Date Columns

### Definition
Date-column identification finds variables that represent dates or timestamps.

### Why this step is important
Credit-risk features often depend on timing, such as account age, payment recency, enquiry recency, and duration between events.

### Objective
Identify columns containing date/time information in all three datasets.

### Expected Output
A list of likely date columns for each table.

### Insight
We will convert only confirmed date fields to datetime rather than blindly converting every object column.

In [ ]:
# Display columns and current data types that may represent dates

for name, df in [("Account", acct), ("Enquiry", enq), ("Demographics", demo)]:
    print(f"\n{name} Data - Possible Date Columns")

    possible_date_cols = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in ["date", "dt", "time"])
    ]

    for col in possible_date_cols:
        print(f"{col:35} | {df[col].dtype}")

# Step 13 — Date Quality Analysis

### Definition
Date quality analysis checks whether date variables contain missing values, invalid values, or unusual date ranges.

### Why this step is important
Incorrect dates can produce incorrect account age, payment recency, enquiry recency, and duration features.

### Objective
Convert confirmed date columns to datetime format and inspect their minimum, maximum, and missing values.

### Expected Output
A date-quality summary containing:
- Column name
- Missing count
- Minimum date
- Maximum date

### Insight
We should identify unusual or invalid date ranges before creating any time-based features.

In [ ]:
# Date columns documented in the project tables

account_date_cols = [
    col for col in [
        "dt_opened",
        "opened_dt",
        "last_paymt_dt",
        "closed_dt",
        "reporting_dt",
        "paymt_str_dt",
        "paymt_end_dt"
    ]
    if col in acct.columns
]

enquiry_date_cols = [
    col for col in [
        "dt_opened",
        "upload_dt",
        "enquiry_dt"
    ]
    if col in enq.columns
]

demo_date_cols = [
    col for col in [
        "dt_opened",
        "entry_time"
    ]
    if col in demo.columns
]

# Convert confirmed date columns
for col in account_date_cols:
    acct[col] = pd.to_datetime(acct[col], errors="coerce")

for col in enquiry_date_cols:
    enq[col] = pd.to_datetime(enq[col], errors="coerce")

for col in demo_date_cols:
    demo[col] = pd.to_datetime(demo[col], errors="coerce")

print("Date conversion completed.")

In [ ]:
def date_quality_summary(df, date_cols):
    rows = []

    for col in date_cols:
        rows.append({
            "Column": col,
            "Missing_Count": df[col].isna().sum(),
            "Missing_Percentage": df[col].isna().mean() * 100,
            "Minimum_Date": df[col].min(),
            "Maximum_Date": df[col].max()
        })

    return pd.DataFrame(rows)

print("ACCOUNT DATE QUALITY")
display(date_quality_summary(acct, account_date_cols))

print("ENQUIRY DATE QUALITY")
display(date_quality_summary(enq, enquiry_date_cols))

print("DEMOGRAPHICS DATE QUALITY")
display(date_quality_summary(demo, demo_date_cols))

# Step 14 — Payment History Analysis

### Definition
Payment-history analysis examines the payment-history variables and their structure.

### Why this step is important
Payment behavior is directly relevant to credit risk. The project also identifies payment-history-derived features among the suggested important features.

### Objective
1. Inspect `paymenthistory1` and `paymenthistory2`.
2. Measure missingness.
3. Examine the typical string/sequence length.
4. Understand whether the fields contain usable payment-history patterns.

### Expected Output
A summary of the payment-history fields including missing values and observed lengths.

### Insight
The payment-history fields can later be transformed into customer-level indicators such as history length or delinquency-related measures, but we should first understand their actual structure.

In [ ]:
payment_history_cols = [
    col for col in ["paymenthistory1", "paymenthistory2"]
    if col in acct.columns
]

payment_history_summary = []

for col in payment_history_cols:
    non_null = acct[col].dropna().astype(str)

    lengths = non_null.str.len()

    payment_history_summary.append({
        "Column": col,
        "Missing_Count": acct[col].isna().sum(),
        "Missing_Percentage": acct[col].isna().mean() * 100,
        "Non_Missing_Count": non_null.shape[0],
        "Min_Length": lengths.min() if len(lengths) else np.nan,
        "Median_Length": lengths.median() if len(lengths) else np.nan,
        "Max_Length": lengths.max() if len(lengths) else np.nan
    })

display(pd.DataFrame(payment_history_summary))

In [ ]:
# Inspect a small sample of payment-history values

for col in payment_history_cols:
    print(f"\nSample values from {col}:")
    display(acct[col].dropna().head(10))

# Step 15 — Customer-Level Relationship Analysis

### Definition
Customer-level relationship analysis checks how the three source tables overlap through the customer identifier.

### Why this step is important
We need to know whether customers in Account and Enquiry data are also present in the Demographics/target population.

### Objective
1. Count unique customers in each table.
2. Measure Account customers present in Demographics.
3. Measure Enquiry customers present in Demographics.
4. Measure customers shared by Account and Enquiry.

### Expected Output
Overlap counts and percentages between the source datasets.

### Insight
This analysis tells us how much historical account and enquiry information is available for the modelling population and helps us design the correct aggregation and merge strategy.

In [ ]:
# Unique customer sets
account_customers = set(acct["customer_no"].dropna().unique())
enquiry_customers = set(enq["customer_no"].dropna().unique())
demo_customers = set(demo["customer_no"].dropna().unique())

# Overlap counts
account_demo_overlap = account_customers & demo_customers
enquiry_demo_overlap = enquiry_customers & demo_customers
account_enquiry_overlap = account_customers & enquiry_customers

relationship_summary = pd.DataFrame({
    "Relationship": [
        "Account customers",
        "Enquiry customers",
        "Demographics customers",
        "Account ∩ Demographics",
        "Enquiry ∩ Demographics",
        "Account ∩ Enquiry"
    ],
    "Unique_Customers": [
        len(account_customers),
        len(enquiry_customers),
        len(demo_customers),
        len(account_demo_overlap),
        len(enquiry_demo_overlap),
        len(account_enquiry_overlap)
    ]
})

display(relationship_summary)

In [ ]:
# Percentage of the Demographics modelling population
demo_count = len(demo_customers)

relationship_percentage = pd.DataFrame({
    "Source": [
        "Account",
        "Enquiry"
    ],
    "Customers_Also_In_Demographics": [
        len(account_demo_overlap),
        len(enquiry_demo_overlap)
    ],
    "Percentage_of_Demographics": [
        len(account_demo_overlap) / demo_count * 100 if demo_count else np.nan,
        len(enquiry_demo_overlap) / demo_count * 100 if demo_count else np.nan
    ]
})

display(relationship_percentage)

# Step 15 — Data Relationship Diagram

This diagram summarizes the source-table relationship before feature engineering.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")

# Positions
account_box = (0.05, 0.55, 0.25, 0.25)
enquiry_box = (0.05, 0.15, 0.25, 0.25)
demo_box = (0.70, 0.35, 0.25, 0.25)
feature_box = (0.38, 0.35, 0.25, 0.25)

for x, y, w, h, label in [
    (*account_box, "Cust_Account\nMany records per customer"),
    (*enquiry_box, "Cust_Enquiry\nMany records per customer"),
    (*feature_box, "Customer-Level\nFeature Matrix"),
    (*demo_box, "Cust_Demographics\nCustomer + Bad_label")
]:
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, linewidth=1.5))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(0.38, 0.55), xytext=(0.30, 0.65),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.38, 0.45), xytext=(0.30, 0.25),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.70, 0.48), xytext=(0.63, 0.48),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))

ax.set_title("Customer-Level Data Relationship", fontsize=14)
plt.show()

### Diagram Insight
The diagram illustrates the modelling architecture: Account and Enquiry histories are first converted from multiple records into customer-level features, while Demographics supplies the customer-level information and target. The three sources then form the modelling feature matrix.

# Step 16 — Clean and Validate Customer Identifiers

### Definition
Customer-identifier cleaning checks whether `customer_no` is present, consistently formatted, and suitable for joining the three source tables.

### Why this step is important
`customer_no` is the link between customer records across Account, Enquiry, and Demographics data. Incorrect or inconsistent identifiers can cause customers to be lost or incorrectly matched during aggregation and merging.

### Objective
- Check missing customer IDs.
- Check data type.
- Check whitespace or formatting issues where applicable.
- Compare the identifier structure across datasets.

### Expected Output
A small quality summary showing missing IDs and the current data type.

### Insight
We should establish a reliable customer key before performing any feature aggregation or merging.

In [ ]:
# Inspect customer_no in each dataset

for name, df in [("Account", acct), ("Enquiry", enq), ("Demographics", demo)]:
    print(f"\n{name} Data")
    print("Data type:", df["customer_no"].dtype)
    print("Missing customer_no:", df["customer_no"].isna().sum())
    print("Unique customer_no:", df["customer_no"].nunique())

    # Show a small sample
    display(df["customer_no"].head())

In [ ]:
# Check whether customer_no contains string whitespace issues
# Only strip whitespace if the column is already textual.

for name, df in [("Account", acct), ("Enquiry", enq), ("Demographics", demo)]:
    if df["customer_no"].dtype == "object":
        stripped = df["customer_no"].astype("string").str.strip()
        whitespace_count = (df["customer_no"].astype("string") != stripped).sum()
    else:
        whitespace_count = 0

    print(f"{name}: potential whitespace records = {whitespace_count}")

# Step 17 — Standardize and Validate Date Fields

### Definition
Date cleaning converts confirmed date columns into a consistent datetime format and identifies missing or suspicious dates.

### Why this step is important
Credit-risk features depend on time, including account duration, payment recency, and enquiry recency.

### Objective
- Confirm datetime types.
- Count missing dates.
- Review minimum and maximum dates.
- Identify dates that may be outside the useful observation period.

### Expected Output
Date ranges and missing-value summaries for the confirmed date fields.

### Insight
We will use only reliable dates when creating duration and recency features.

In [ ]:
# Review the date columns after conversion from Step 13

print("ACCOUNT DATE COLUMNS")
for col in account_date_cols:
    print(
        f"{col}: dtype={acct[col].dtype}, "
        f"missing={acct[col].isna().sum()}, "
        f"min={acct[col].min()}, "
        f"max={acct[col].max()}"
    )

print("\nENQUIRY DATE COLUMNS")
for col in enquiry_date_cols:
    print(
        f"{col}: dtype={enq[col].dtype}, "
        f"missing={enq[col].isna().sum()}, "
        f"min={enq[col].min()}, "
        f"max={enq[col].max()}"
    )

print("\nDEMOGRAPHICS DATE COLUMNS")
for col in demo_date_cols:
    print(
        f"{col}: dtype={demo[col].dtype}, "
        f"missing={demo[col].isna().sum()}, "
        f"min={demo[col].min()}, "
        f"max={demo[col].max()}"
    )

# Step 18 — Account-Level Feature Engineering

### Definition
Account-level feature engineering converts multiple account records into meaningful customer-level summaries.

### Why this step is important
The Account table contains historical account and payment information, and a customer can have multiple account records. A machine-learning dataset should eventually contain one modelling record per customer.

### Candidate Features
Examples include:
- Number of accounts.
- Total credit limit.
- Total cash limit.
- Total current balance.
- Total amount past due.
- Average interest rate.
- Number of accounts with past-due amounts.

### Important Note
These are initial features for exploration. We will validate their distributions and business meaning before final feature selection.

### Expected Output
One row per `customer_no` in the aggregated Account feature table.

### Insight
Aggregation changes the Account table from **account-level grain** to **customer-level grain**, making it suitable for joining with customer-level data.

In [ ]:

# Create a copy
acct_features_source = acct.copy()

# Convert important Account columns to numeric
numeric_account_cols = [
    "high_credit_amt",
    "cur_balance_amt",
    "amt_past_due",
    "creditlimit",
    "cashlimit",
    "rateofinterest"
]

for col in numeric_account_cols:
    if col in acct_features_source.columns:
        acct_features_source[col] = pd.to_numeric(
            acct_features_source[col],
            errors="coerce"
        )

# Create customer-level Account features
account_features = (
    acct_features_source
    .groupby("customer_no")
    .agg(
        account_count=("customer_no", "size"),
        total_high_credit_amt=("high_credit_amt", "sum"),
        total_current_balance=("cur_balance_amt", "sum"),
        total_amount_past_due=("amt_past_due", "sum"),
        total_credit_limit=("creditlimit", "sum"),
        total_cash_limit=("cashlimit", "sum"),
        mean_interest_rate=("rateofinterest", "mean")
    )
    .reset_index()
)

# Create past-due account indicator
past_due_by_customer = (
    acct_features_source
    .assign(
        past_due_flag=(
            acct_features_source["amt_past_due"]
            .fillna(0) > 0
        ).astype(int)
    )
    .groupby("customer_no")["past_due_flag"]
    .sum()
    .rename("accounts_with_past_due")
    .reset_index()
)

# Merge past-due feature
account_features = account_features.merge(
    past_due_by_customer,
    on="customer_no",
    how="left"
)

print("Account feature matrix created successfully.")
print("Shape:", account_features.shape)
print("Unique customers:", account_features["customer_no"].nunique())

display(account_features.head())

# Step 19 — Enquiry-Level Feature Engineering

### Definition
Enquiry-level feature engineering converts multiple credit-enquiry records into customer-level summary variables.

### Why this step is important
The project states that the Enquiry table contains historical enquiry information such as enquiry amount and enquiry purpose. Customers can have multiple enquiries, so the raw table must be summarized before modelling.

### Candidate Features
Examples include:
- Total number of enquiries.
- Total enquiry amount.
- Average enquiry amount.
- Most recent enquiry date.
- Number of enquiries in recent periods.

### Expected Output
One row per `customer_no` in the aggregated Enquiry feature table.

### Insight
Enquiry behavior can be represented as customer-level numerical features rather than retaining multiple enquiry rows per customer.

In [ ]:
# Step 19 — Enquiry Feature Engineering

# Create a copy
enq_features_source = enq.copy()

# Convert enquiry amount to numeric
enq_features_source["enq_amt"] = pd.to_numeric(
    enq_features_source["enq_amt"],
    errors="coerce"
)

# Convert enquiry date to datetime
enq_features_source["enquiry_dt"] = pd.to_datetime(
    enq_features_source["enquiry_dt"],
    errors="coerce"
)

# Create customer-level enquiry features
enquiry_features = (
    enq_features_source
    .groupby("customer_no")
    .agg(
        enquiry_count=("customer_no", "size"),
        total_enquiry_amount=("enq_amt", "sum"),
        mean_enquiry_amount=("enq_amt", "mean"),
        first_enquiry_date=("enquiry_dt", "min"),
        last_enquiry_date=("enquiry_dt", "max")
    )
    .reset_index()
)

# Reference date = latest enquiry date available in the dataset
reference_date = enq_features_source["enquiry_dt"].max()

# Calculate days since last enquiry
enquiry_features["days_since_last_enquiry"] = (
    reference_date - enquiry_features["last_enquiry_date"]
).dt.days

print("Enquiry feature matrix created successfully.")
print("Reference date:", reference_date)
print("Shape:", enquiry_features.shape)
print(
    "Unique customers:",
    enquiry_features["customer_no"].nunique()
)

display(enquiry_features.head())

# Step 20 — Create the Initial Customer Feature Matrix

### Definition
The customer feature matrix combines customer-level Account features, Enquiry features, and the Demographics data into one modelling dataset.

### Why this step is important
Machine-learning algorithms require a modelling table where each row represents the unit being predicted. Here, that unit is the customer.

### Objective
- Keep one row per customer.
- Join Account features using `customer_no`.
- Join Enquiry features using `customer_no`.
- Retain Demographics and `Bad_label`.
- Check the final shape and target availability.

### Expected Output
A single customer-level DataFrame with:
- One row per customer.
- Account-derived features.
- Enquiry-derived features.
- Demographic/application features.
- `Bad_label` as the target.

### Insight
This is the first version of the **customer-level feature matrix** requested in the project brief. It is still an initial feature matrix; we will continue cleaning, exploring, and selecting features before modelling.

In [ ]:
# Start from the customer-level Demographics table
model_data = demo.copy()

# Check that customer_no is unique in the customer-level table
print("Demographics rows:", len(model_data))
print("Unique customers:", model_data["customer_no"].nunique())

# Merge Account features
model_data = model_data.merge(
    account_features,
    on="customer_no",
    how="left"
)

# Merge Enquiry features
model_data = model_data.merge(
    enquiry_features,
    on="customer_no",
    how="left"
)

print("Initial feature matrix shape:", model_data.shape)
display(model_data.head())

In [ ]:
# Validate the customer-level grain

print("Rows in feature matrix       :", len(model_data))
print("Unique customers             :", model_data["customer_no"].nunique())
print("Duplicate customer rows      :", model_data["customer_no"].duplicated().sum())
print("Target column present        :", "Bad_label" in model_data.columns)

print("\nTarget distribution:")
display(model_data["Bad_label"].value_counts(dropna=False))

# Step 20 — Feature Matrix Construction Diagram

This diagram shows how the three sources are transformed into a single customer-level modelling dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.axis("off")

items = [
    (0.03, 0.60, 0.22, 0.20, "Cust_Account\nMultiple records/customer"),
    (0.03, 0.20, 0.22, 0.20, "Cust_Enquiry\nMultiple records/customer"),
    (0.39, 0.40, 0.24, 0.22, "Customer-Level\nAggregation"),
    (0.75, 0.40, 0.22, 0.22, "Initial Feature Matrix\n+ Bad_label")
]

for x, y, w, h, label in items:
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, linewidth=1.5))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(0.39, 0.51), xytext=(0.25, 0.67),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.39, 0.47), xytext=(0.25, 0.30),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.annotate("", xy=(0.75, 0.51), xytext=(0.63, 0.51),
            arrowprops=dict(arrowstyle="->", linewidth=1.5))

ax.set_title("Initial Customer-Level Feature Matrix Construction", fontsize=14)
plt.show()

### Diagram Insight
The diagram represents the key transformation in this stage: multiple Account and Enquiry records are summarized at customer level, then combined with the customer-level Demographics data and target to create the initial feature matrix.

# Step 21 — Validate the Customer Feature Matrix

### Definition
Feature-matrix validation checks whether the combined `model_data` table has the correct customer-level structure.

### Why this step is important
The machine-learning dataset should represent the modelling population at the customer level, with one row per customer.

### Objective
Check:
- Number of rows and columns.
- Number of unique customers.
- Duplicate customer records.
- Presence of `Bad_label`.
- Target missing values.

### Expected Output
A structural summary of `model_data`.

### Insight
If the number of rows equals the number of unique customers and there are no duplicate customer IDs, the feature matrix has the correct customer-level grain.

In [ ]:
# Step 21 — Validate the feature matrix

print("Feature Matrix Shape:", model_data.shape)
print("Number of Rows:", len(model_data))
print("Unique Customers:", model_data["customer_no"].nunique())
print("Duplicate Customer Rows:", model_data["customer_no"].duplicated().sum())
print("Target Column Present:", "Bad_label" in model_data.columns)
print("Missing Target Values:", model_data["Bad_label"].isna().sum())

print("\nTarget Distribution:")
display(model_data["Bad_label"].value_counts(dropna=False))

# Step 22 — Missing Values in the Feature Matrix

### Definition
This step measures missing values after Account and Enquiry features have been merged into the customer-level dataset.

### Why this step is important
A missing aggregated value can mean either:
- the customer had no corresponding history in that source table, or
- the underlying information was genuinely missing.

These two situations should not automatically be treated the same way.

### Objective
Calculate missing counts and percentages for all features.

### Expected Output
A sorted missing-value table.

### Insight
This step helps us distinguish between **absence of history** and actual missing information before deciding on imputation rules.

In [ ]:
# Step 22 — Missing values in model_data

missing_features = pd.DataFrame({
    "Missing_Count": model_data.isna().sum(),
    "Missing_Percentage": model_data.isna().mean() * 100
})

missing_features = (
    missing_features
    .sort_values("Missing_Percentage", ascending=False)
)

display(missing_features)

In [ ]:
# Show only columns with missing values

missing_only = missing_features[
    missing_features["Missing_Count"] > 0
]

print("Number of features containing missing values:",
      len(missing_only))

display(missing_only)

# Step 23 — Numerical Feature Distribution and Outlier Analysis

### Definition
Distribution analysis examines the statistical behavior of numerical variables, including central tendency, spread, skewness, and extreme values.

### Why this step is important
Credit-risk variables such as balances, credit limits, overdue amounts, and enquiry amounts can have highly skewed distributions.

### Objective
Calculate descriptive statistics and skewness for numerical features.

### Expected Output
A numerical summary containing:
- Count
- Mean
- Standard deviation
- Minimum
- Median
- Maximum
- Skewness

### Insight
Highly skewed variables may need transformation, capping, or other treatment depending on the model and business interpretation.

In [ ]:
# Step 23 — Numerical feature summary

numeric_cols = model_data.select_dtypes(
    include=np.number
).columns.tolist()

numeric_summary = model_data[numeric_cols].describe().T

numeric_summary["median"] = model_data[numeric_cols].median()
numeric_summary["skewness"] = model_data[numeric_cols].skew()

numeric_summary = numeric_summary[
    [
        "count",
        "mean",
        "std",
        "min",
        "median",
        "max",
        "skewness"
    ]
]

display(numeric_summary.sort_values(
    "skewness",
    key=lambda x: x.abs(),
    ascending=False
))

In [ ]:
# Plot distributions for selected initial Account/Enquiry features

plot_cols = [
    col for col in [
        "account_count",
        "total_current_balance",
        "total_amount_past_due",
        "total_credit_limit",
        "total_enquiry_amount",
        "enquiry_count"
    ]
    if col in model_data.columns
]

for col in plot_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(
        data=model_data,
        x=col,
        kde=True
    )
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# Step 24 — Compare Important Features by Credit-Risk Class

### Definition
This analysis compares candidate numerical features between `Bad_label = 0` and `Bad_label = 1`.

### Why this step is important
It provides an initial business-level understanding of whether customer account and enquiry behavior differs between the two target groups.

### Candidate Variables
Where available:
- `total_current_balance`
- `total_credit_limit`
- `total_amount_past_due`
- `account_count`
- `enquiry_count`
- `total_enquiry_amount`
- `mean_enquiry_amount`

### Objective
Compare distributions using summary statistics and boxplots.

### Expected Output
Class-wise summary statistics and visual comparisons.

### Insight
A visible difference between the good and bad groups indicates that a variable may contain predictive information, but this alone does not prove that it should be included in the final model.

In [ ]:
# Step 24 — Class-wise summary statistics

candidate_cols = [
    col for col in [
        "account_count",
        "total_current_balance",
        "total_credit_limit",
        "total_amount_past_due",
        "total_enquiry_amount",
        "mean_enquiry_amount",
        "enquiry_count"
    ]
    if col in model_data.columns
]

class_summary = (
    model_data
    .groupby("Bad_label")[candidate_cols]
    .agg(["count", "mean", "median"])
)

display(class_summary)

In [ ]:
# Boxplots for candidate features

for col in candidate_cols:
    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=model_data,
        x="Bad_label",
        y=col
    )

    plt.title(f"{col} by Bad_label")
    plt.xlabel("Bad_label")
    plt.ylabel(col)
    plt.show()

# Step 25 — Initial Feature Screening

### Definition
Initial feature screening removes or flags variables that are clearly unsuitable for modelling before formal feature selection.

### Why this step is important
Identifiers, target variables, constant columns, and unsuitable object/date fields should not be treated as ordinary predictive variables.

### Screening Rules
We will identify:
1. Customer identifiers.
2. Target variable.
3. Constant columns.
4. High-missingness columns.
5. Object/date columns requiring transformation.

### Expected Output
Separate lists of columns for each screening category.

### Insight
This is an initial screening stage only. We are not yet selecting the final predictors; that will happen after deeper EDA and feature-quality analysis.

In [ ]:
# Step 25 — Initial feature screening

# Identifier and target
identifier_cols = [
    col for col in ["customer_no"]
    if col in model_data.columns
]

target_col = "Bad_label"

# Constant columns
constant_cols = [
    col for col in model_data.columns
    if model_data[col].nunique(dropna=False) <= 1
]

# High-missingness columns: threshold used only for screening
high_missing_cols = [
    col for col in model_data.columns
    if model_data[col].isna().mean() > 0.90
]

# Object and datetime columns
object_cols = model_data.select_dtypes(
    include=["object", "string"]
).columns.tolist()

datetime_cols = model_data.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns.tolist()

print("Identifier columns:")
print(identifier_cols)

print("\nTarget column:")
print(target_col)

print("\nConstant columns:")
print(constant_cols)

print("\nHigh-missingness columns (>90%):")
print(high_missing_cols)

print("\nObject/String columns:")
print(object_cols)

print("\nDatetime columns:")
print(datetime_cols)

In [ ]:
# Build an initial candidate feature list
excluded_from_initial_model = set(identifier_cols + [target_col])

candidate_features = [
    col for col in model_data.columns
    if col not in excluded_from_initial_model
    and col not in constant_cols
]

print("Total columns in model_data:", model_data.shape[1])
print("Initial candidate features:", len(candidate_features))

print("\nFirst 50 candidate features:")
print(candidate_features[:50])

# Step 26 — Payment-History Features

### Definition
Payment-history feature engineering converts the raw payment-history fields into numerical customer-level variables.

### Why this step is important
Payment behavior is directly related to credit risk, and the project document specifically suggests payment-history features such as `payment_history_mean_length` and `payment_history_avg_dpd_0_29_bucket`.

### Objective
1. Inspect the raw payment-history fields.
2. Calculate payment-history length where possible.
3. Aggregate the length at customer level.
4. Preserve the raw values for further investigation.

### Expected Output
Customer-level payment-history length features.

### Insight
We can safely calculate history length from the observed data structure, but we should only calculate DPD-bucket features after confirming how the payment-history codes are represented.

In [ ]:
# Step 26 — Payment-history length features

payment_history_cols = [
    col for col in ["paymenthistory1", "paymenthistory2"]
    if col in acct_features_source.columns
]

payment_history_work = acct_features_source[
    ["customer_no"] + payment_history_cols
].copy()

for col in payment_history_cols:
    payment_history_work[f"{col}_length"] = (
        payment_history_work[col]
        .astype("string")
        .str.strip()
        .replace({"<NA>": pd.NA, "nan": pd.NA})
        .str.len()
    )

length_cols = [
    f"{col}_length"
    for col in payment_history_cols
]

if length_cols:
    payment_history_work["payment_history_length"] = (
        payment_history_work[length_cols]
        .mean(axis=1)
    )

    payment_history_features = (
        payment_history_work
        .groupby("customer_no")
        .agg(
            payment_history_mean_length=(
                "payment_history_length", "mean"
            )
        )
        .reset_index()
    )
else:
    payment_history_features = pd.DataFrame(
        columns=["customer_no", "payment_history_mean_length"]
    )

print("Payment-history feature shape:",
      payment_history_features.shape)

display(payment_history_features.head())

# Step 27 — Account Duration Features

### Definition
Account-duration features measure the time between important account events, such as account opening and the latest payment.

### Why this step is important
The project specifically suggests `total_diff_lastpaymt_opened_dt` and `mean_diff_lastpaymt_opened_dt`.

### Objective
Calculate customer-level duration features from:
- Account opening date.
- Last payment date.

### Expected Output
Customer-level total and average account/payment duration features.

### Insight
Raw dates are converted into interpretable numerical durations that can be used directly by machine-learning models.

In [ ]:
# Step 27 — Account duration features

duration_source = acct_features_source.copy()

duration_source["opened_dt"] = pd.to_datetime(
    duration_source["opened_dt"],
    errors="coerce"
)

duration_source["last_paymt_dt"] = pd.to_datetime(
    duration_source["last_paymt_dt"],
    errors="coerce"
)

duration_source["diff_lastpaymt_opened_dt"] = (
    duration_source["last_paymt_dt"]
    - duration_source["opened_dt"]
).dt.days

account_duration_features = (
    duration_source
    .groupby("customer_no")
    .agg(
        total_diff_lastpaymt_opened_dt=(
            "diff_lastpaymt_opened_dt", "sum"
        ),
        mean_diff_lastpaymt_opened_dt=(
            "diff_lastpaymt_opened_dt", "mean"
        )
    )
    .reset_index()
)

display(account_duration_features.head())

print(
    "Account duration feature shape:",
    account_duration_features.shape
)

# Step 28 — Credit Utilization Features

### Definition
Credit utilization measures the relationship between the customer's outstanding balance and available credit.

### Why this step is important
The project specifically suggests `Ratio_currbalance_creditlimit` and `utilisation_trend`.

### Objective
Calculate customer-level:
- Total current balance.
- Total credit limit.
- Current-balance-to-credit-limit ratio.
- A utilization trend proxy using available balance and cash limit.

### Expected Output
Customer-level utilization variables.

### Insight
Higher credit utilization can represent greater exposure relative to available credit, but the relationship with the target must be evaluated empirically rather than assumed.

In [ ]:
# Step 28 — Credit utilization features

utilization_features = account_features[
    [
        "customer_no",
        "total_current_balance",
        "total_credit_limit",
        "total_cash_limit"
    ]
].copy()

# Avoid division by zero
utilization_features["Ratio_currbalance_creditlimit"] = np.where(
    utilization_features["total_credit_limit"] != 0,
    utilization_features["total_current_balance"]
    / utilization_features["total_credit_limit"],
    np.nan
)

# A customer-level utilization measure incorporating cash limit
denominator = (
    utilization_features["total_credit_limit"].fillna(0)
    + utilization_features["total_cash_limit"].fillna(0)
)

utilization_features["utilisation_trend"] = np.where(
    denominator != 0,
    utilization_features["total_current_balance"]
    / denominator,
    np.nan
)

display(
    utilization_features[
        [
            "customer_no",
            "total_current_balance",
            "total_credit_limit",
            "total_cash_limit",
            "Ratio_currbalance_creditlimit",
            "utilisation_trend"
        ]
    ].head()
)

# Step 29 — Enquiry Recency Features

### Definition
Create customer-level features that measure recent credit-enquiry activity, including enquiry counts over the last 90 and 365 days and the timing relationship between enquiry date and application/opening date.

### Why this step is important
Recent enquiries can provide information about current credit demand. The project specifically lists 90-day and 365-day enquiry counts among its suggested features.

### Expected Output
Customer-level enquiry-recency features such as `count_enquiry_recency_90`, `count_enquiry_recency_365`, `mean_diff_open_enquiry_dt`, and enquiry-purpose information where available.

### Insight
Transaction-level enquiry history is converted into customer-level risk indicators that can be joined to the final modelling matrix.

In [ ]:
# Step 29 — Enquiry recency features

enquiry_work = enq_features_source.copy()

reference_date = enquiry_work["enquiry_dt"].max()

# 90-day and 365-day cutoffs
cutoff_90 = reference_date - pd.Timedelta(days=90)
cutoff_365 = reference_date - pd.Timedelta(days=365)

enquiry_work["is_recent_90"] = (
    enquiry_work["enquiry_dt"].between(
        cutoff_90, reference_date
    )
).astype(int)

enquiry_work["is_recent_365"] = (
    enquiry_work["enquiry_dt"].between(
        cutoff_365, reference_date
    )
).astype(int)

recency_features = (
    enquiry_work
    .groupby("customer_no")
    .agg(
        count_enquiry_recency_90=("is_recent_90", "sum"),
        count_enquiry_recency_365=("is_recent_365", "sum")
    )
    .reset_index()
)

# Mean difference between dt_opened and enquiry date if both are available
if "dt_opened" in enquiry_work.columns:
    enquiry_work["dt_opened"] = pd.to_datetime(
        enquiry_work["dt_opened"],
        errors="coerce"
    )

    enquiry_work["mean_diff_open_enquiry_dt_value"] = (
        enquiry_work["enquiry_dt"]
        - enquiry_work["dt_opened"]
    ).dt.days

    mean_open_enquiry = (
        enquiry_work
        .groupby("customer_no")[
            "mean_diff_open_enquiry_dt_value"
        ]
        .mean()
        .rename("mean_diff_open_enquiry_dt")
        .reset_index()
    )

    recency_features = recency_features.merge(
        mean_open_enquiry,
        on="customer_no",
        how="left"
    )

# Most frequent enquiry purpose
if "enq_purpose" in enquiry_work.columns:
    max_freq_enquiry = (
        enquiry_work
        .groupby("customer_no")["enq_purpose"]
        .agg(
            lambda x: x.mode().iloc[0]
            if not x.mode().empty else np.nan
        )
        .rename("max_freq_enquiry")
        .reset_index()
    )

    recency_features = recency_features.merge(
        max_freq_enquiry,
        on="customer_no",
        how="left"
    )

print("Reference date:", reference_date)
print("Enquiry-recency feature shape:", recency_features.shape)

display(recency_features.head())

# Step 30 — Combine Engineered Features and Perform Feature Selection Screening

### Definition
This step combines the engineered features and performs an initial quality screen before model development.

### Why this step is important
Feature engineering creates candidate predictors, but not every variable should enter the final model. We need to check duplicates, missingness, constants, data types, and target leakage.

### Objective
1. Merge the new engineered features into `model_data`.
2. Remove duplicate feature columns created by earlier stages.
3. Identify constant variables.
4. Identify highly missing variables.
5. Separate identifiers and target from predictors.

### Expected Output
An updated customer-level feature matrix and an initial candidate-feature list.

### Insight
This produces a richer candidate feature matrix while keeping the final model selection open for later statistical and model-based evaluation.

In [ ]:
# Step 30 — Combine engineered features

# Start from the current customer-level feature matrix
model_data = model_data.copy()

# Merge engineered feature groups
feature_groups = [
    payment_history_features,
    account_duration_features,
    utilization_features,
    recency_features
]

for features in feature_groups:
    if not features.empty:
        new_cols = [
            col for col in features.columns
            if col != "customer_no"
            and col not in model_data.columns
        ]

        model_data = model_data.merge(
            features[["customer_no"] + new_cols],
            on="customer_no",
            how="left"
        )

print("Updated feature matrix shape:", model_data.shape)
print(
    "Unique customers:",
    model_data["customer_no"].nunique()
)

display(model_data.head())

In [ ]:
# Initial feature-quality screening

constant_cols = [
    col for col in model_data.columns
    if model_data[col].nunique(dropna=False) <= 1
]

high_missing_cols = [
    col for col in model_data.columns
    if model_data[col].isna().mean() > 0.90
]

identifier_cols = [
    col for col in ["customer_no"]
    if col in model_data.columns
]

excluded_cols = set(identifier_cols + ["Bad_label"])

candidate_features = [
    col for col in model_data.columns
    if col not in excluded_cols
    and col not in constant_cols
]

print("Constant columns:")
print(constant_cols)

print("\nHigh-missingness columns (>90%):")
print(high_missing_cols)

print("\nNumber of candidate features:", len(candidate_features))

print("\nCandidate feature sample:")
print(candidate_features[:50])

# Step 31 — Target Leakage Check

### Definition
Target leakage occurs when a predictor contains information that would not be legitimately available at the time the model makes its prediction, or directly contains information derived from the target.

### Why this step is important
Leakage can produce unrealistically strong model performance and make the model unsuitable for real-world credit-risk use.

### Objective
Identify:
- The target variable.
- Customer identifiers.
- Variables whose names may indicate direct target information.
- Variables that need business-time validation.

### Expected Output
A list of columns requiring leakage investigation.

### Insight
We should remove a variable only when there is evidence that it is unavailable at prediction time or directly leaks the target.

In [ ]:
# Step 31 — Initial target leakage screening

target_col = "Bad_label"
id_cols = [col for col in ["customer_no"] if col in model_data.columns]

# Names that deserve investigation, not automatic removal
leakage_keywords = [
    "bad_label",
    "target",
    "default",
    "dpd_30",
    "30_plus",
    "delinquen"
]

potential_leakage_cols = [
    col for col in model_data.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

print("Target column:", target_col)
print("Identifier columns:", id_cols)

print("\nPotential leakage columns requiring investigation:")
print(potential_leakage_cols)

# Step 32 — Categorical Feature Treatment

### Definition
Categorical variables contain labels or groups rather than continuous numerical measurements.

### Why this step is important
Many machine-learning algorithms require numerical predictors, so categorical variables need an appropriate representation.

### Objective
Identify categorical columns and inspect their cardinality.

### Expected Output
A table showing categorical columns and their number of unique values.

### Insight
Low-cardinality categorical variables may be suitable for one-hot encoding, while extremely high-cardinality variables may require a different strategy or exclusion.

In [ ]:
# Step 32 — Identify categorical columns

categorical_cols = model_data.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_summary = pd.DataFrame({
    "Column": categorical_cols,
    "Unique_Values": [
        model_data[col].nunique(dropna=True)
        for col in categorical_cols
    ],
    "Missing_Count": [
        model_data[col].isna().sum()
        for col in categorical_cols
    ]
})

display(categorical_summary.sort_values(
    "Unique_Values",
    ascending=False
))

# Step 33 — Correlation and Multicollinearity Analysis

### Definition
Correlation measures the strength of the relationship between numerical variables.

### Why this step is important
Highly correlated features can contain redundant information and make feature interpretation more difficult.

### Objective
1. Calculate correlations among numerical candidate predictors.
2. Visualize the correlation structure.
3. Identify highly correlated feature pairs.

### Expected Output
A correlation matrix, heatmap, and a list of highly correlated feature pairs.

### Insight
Correlation does not prove causation or predictive value. It is used here as a screening tool to identify redundancy.

In [ ]:
# Step 33 — Numerical correlation matrix

numeric_candidate_cols = [
    col for col in candidate_features
    if col in model_data.columns
    and pd.api.types.is_numeric_dtype(model_data[col])
]

corr_matrix = model_data[numeric_candidate_cols].corr()

display(corr_matrix)

In [ ]:
# Correlation heatmap

if len(numeric_candidate_cols) > 1:
    plt.figure(figsize=(14, 10))

    sns.heatmap(
        corr_matrix,
        cmap="coolwarm",
        center=0
    )

    plt.title("Correlation Heatmap — Candidate Numerical Features")
    plt.show()
else:
    print("Not enough numerical candidate features for a correlation heatmap.")

In [ ]:
# Identify highly correlated feature pairs

correlation_threshold = 0.80

corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]

        if abs(corr_value) >= correlation_threshold:
            corr_pairs.append({
                "Feature_1": corr_matrix.columns[i],
                "Feature_2": corr_matrix.columns[j],
                "Correlation": corr_value
            })

high_corr_pairs = pd.DataFrame(corr_pairs)

if high_corr_pairs.empty:
    print("No feature pairs found above the correlation threshold.")
else:
    high_corr_pairs = high_corr_pairs.sort_values(
        "Correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
    display(high_corr_pairs)

# Step 34 — Missing-Value Treatment

### Definition
Missing-value treatment prepares predictor variables so that the modelling algorithm can process them appropriately.

### Why this step is important
Many machine-learning algorithms cannot directly handle missing numerical values.

### Important Business Rule
For aggregated Account or Enquiry features, a missing value can sometimes mean the customer had **no corresponding history**, rather than that the value was unknown.

Therefore, we first inspect the missingness before applying a treatment.

### Objective
Create a modelling copy and apply conservative numerical imputation.

### Expected Output
A modelling matrix with no missing numerical predictor values.

### Insight
The treatment should be documented and applied using statistics calculated from the training data in a production-quality workflow.

In [ ]:
# Step 34 — Prepare numerical modelling features

# Exclude target and identifiers
model_feature_cols = [
    col for col in model_data.columns
    if col not in id_cols + [target_col]
]

# Keep numerical features for the initial modelling matrix
model_numeric_cols = [
    col for col in model_feature_cols
    if pd.api.types.is_numeric_dtype(model_data[col])
]

X_numeric = model_data[model_numeric_cols].copy()
y = model_data[target_col].copy()

print("Number of numerical predictors:", len(model_numeric_cols))
print("Missing numerical values before treatment:",
      X_numeric.isna().sum().sum())

# Median imputation for initial modelling preparation
# This is a preparation step; production modelling should fit
# imputation statistics on training data only.
for col in X_numeric.columns:
    median_value = X_numeric[col].median()

    if pd.notna(median_value):
        X_numeric[col] = X_numeric[col].fillna(median_value)

print(
    "Missing numerical values after treatment:",
    X_numeric.isna().sum().sum()
)

# Step 35 — Train/Test Split and Final Modelling Matrix

### Definition
A train/test split separates the data into a training population used to fit the model and a test population reserved for final evaluation.

### Why this step is important
The model must be evaluated on observations it did not use during training.

### Objective
- Separate `X` predictors from `y = Bad_label`.
- Use a stratified split so the target-class proportions are approximately preserved.
- Fix a random state for reproducibility.
- Confirm the resulting shapes and class distributions.

### Expected Output
Training and testing datasets with their respective dimensions and target distributions.

### Insight
The test set should remain untouched during feature selection and model tuning as much as possible so that the final performance estimate is meaningful.

In [ ]:
# Step 35 — Train/Test Split

from sklearn.model_selection import train_test_split

X = X_numeric.copy()
y = model_data[target_col].copy()

# Remove rows where the target is missing
valid_target_mask = y.notna()

X = X.loc[valid_target_mask].copy()
y = y.loc[valid_target_mask].copy()

# Convert target to integer when possible
y = pd.to_numeric(y, errors="coerce")

# Remove any rows that became invalid after conversion
valid_target_mask = y.notna()

X = X.loc[valid_target_mask].copy()
y = y.loc[valid_target_mask].astype(int)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

In [ ]:
# Compare target distributions

train_distribution = (
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename("Train_Percentage")
)

test_distribution = (
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename("Test_Percentage")
)

target_comparison = pd.concat(
    [train_distribution, test_distribution],
    axis=1
)

display(target_comparison)

# Step 36 — Rebuild the Full Customer-Level Modelling Matrix

### Definition
Build a one-row-per-customer modelling dataset using the full demographic/application table plus engineered Account, payment-history and Enquiry features.

### Why this step is important
The final modelling matrix should retain the anonymized `feature_1`–`feature_79` variables as well as engineered credit-risk predictors. The project document states that these demographic features are anonymized/obscured for privacy.

### Expected Output
A customer-level matrix containing the target, anonymized application features, engineered credit-risk features, and no customer identifier in the predictor set.

### Insight
This is the principal modelling matrix used for benchmark-oriented model development.

In [ ]:
# Step 36 — Rebuild a full customer-level modelling matrix

import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier

RANDOM_STATE = 42
BENCHMARK_GINI = 0.379
TARGET = "Bad_label"

# Work from the original three tables already loaded in the earlier notebook steps.
demo_work = demo.copy()
acct_work = acct.copy()
enq_work = enq.copy()

# Standardize customer key where necessary.
for df in [demo_work, acct_work, enq_work]:
    if "customer_no" in df.columns:
        if df["customer_no"].dtype == "object":
            df["customer_no"] = df["customer_no"].astype("string").str.strip()

# ---------- Account aggregations ----------
for col in [
    "high_credit_amt", "cur_balance_amt", "amt_past_due",
    "creditlimit", "cashlimit", "rateofinterest", "actualpaymentamount"
]:
    if col in acct_work.columns:
        acct_work[col] = pd.to_numeric(acct_work[col], errors="coerce")

for col in ["opened_dt", "last_paymt_dt", "closed_dt", "reporting_dt",
            "paymt_str_dt", "paymt_end_dt", "dt_opened", "upload_dt"]:
    if col in acct_work.columns:
        acct_work[col] = pd.to_datetime(acct_work[col], errors="coerce")

account_base = (
    acct_work.groupby("customer_no")
    .agg(
        account_count=("customer_no", "size"),
        total_high_credit_amt=("high_credit_amt", "sum"),
        total_current_balance=("cur_balance_amt", "sum"),
        total_amount_past_due=("amt_past_due", "sum"),
        total_credit_limit=("creditlimit", "sum"),
        total_cash_limit=("cashlimit", "sum"),
        mean_interest_rate=("rateofinterest", "mean"),
        total_actual_payment=("actualpaymentamount", "sum")
    )
    .reset_index()
)

# Past-due account count
past_due = (
    acct_work.assign(
        past_due_flag=(acct_work["amt_past_due"].fillna(0) > 0).astype(int)
    )
    .groupby("customer_no")["past_due_flag"]
    .agg(["sum", "mean"])
    .rename(columns={"sum": "accounts_with_past_due",
                     "mean": "past_due_account_ratio"})
    .reset_index()
)

account_base = account_base.merge(past_due, on="customer_no", how="left")

# Account duration
acct_work["account_age_days"] = (
    acct_work["last_paymt_dt"] - acct_work["opened_dt"]
).dt.days

duration = (
    acct_work.groupby("customer_no")
    .agg(
        total_diff_lastpaymt_opened_dt=("account_age_days", "sum"),
        mean_diff_lastpaymt_opened_dt=("account_age_days", "mean"),
        max_account_age_days=("account_age_days", "max")
    )
    .reset_index()
)

account_base = account_base.merge(duration, on="customer_no", how="left")

# Utilization
account_base["Ratio_currbalance_creditlimit"] = np.where(
    account_base["total_credit_limit"].abs() > 0,
    account_base["total_current_balance"] / account_base["total_credit_limit"],
    np.nan
)

util_denominator = (
    account_base["total_credit_limit"].fillna(0)
    + account_base["total_cash_limit"].fillna(0)
)

account_base["utilisation_trend"] = np.where(
    util_denominator.abs() > 0,
    account_base["total_current_balance"] / util_denominator,
    np.nan
)

# ---------- Payment-history features ----------
payment_cols = [
    c for c in ["paymenthistory1", "paymenthistory2"]
    if c in acct_work.columns
]

PAYMENT_TOKEN_PATTERN = re.compile(r"\d{3}|STD|SUB|DBT|LSS|XXX|DDD")

def parse_payment_history(value):
    """
    Extract numeric DPD observations from a payment-history string.
    Numeric values are retained as DPD days. Standard non-numeric codes
    are retained through separate indicators below.

    This is deliberately conservative: it does not invent a DPD value for
    non-numeric codes.
    """
    if pd.isna(value):
        return []
    s = str(value).strip().upper()
    # Payment histories commonly contain 3-character tokens.
    tokens = PAYMENT_TOKEN_PATTERN.findall(s)
    numeric = []
    for token in tokens:
        if token.isdigit():
            numeric.append(int(token))
    return numeric

payment_work = acct_work[["customer_no"] + payment_cols].copy()

if payment_cols:
    for col in payment_cols:
        payment_work[f"{col}_numeric"] = payment_work[col].map(parse_payment_history)

    all_lists = []
    for _, row in payment_work.iterrows():
        vals = []
        for col in payment_cols:
            vals.extend(row[f"{col}_numeric"])
        all_lists.append(vals)

    payment_work["dpd_values"] = all_lists

    payment_work["payment_history_length"] = payment_work["dpd_values"].map(len)
    payment_work["dpd_0_29_count"] = payment_work["dpd_values"].map(
        lambda x: sum(0 <= v <= 29 for v in x)
    )
    payment_work["dpd_30_plus_count"] = payment_work["dpd_values"].map(
        lambda x: sum(v >= 30 for v in x)
    )
    payment_work["max_numeric_dpd"] = payment_work["dpd_values"].map(
        lambda x: max(x) if x else np.nan
    )
    payment_work["mean_numeric_dpd"] = payment_work["dpd_values"].map(
        lambda x: np.mean(x) if x else np.nan
    )
    payment_work["has_numeric_30_plus"] = (
        payment_work["dpd_30_plus_count"] > 0
    ).astype(int)

    payment_features = (
        payment_work.groupby("customer_no")
        .agg(
            payment_history_mean_length=("payment_history_length", "mean"),
            payment_history_avg_dpd_0_29_bucket=("dpd_0_29_count", "mean"),
            payment_history_avg_30_plus_count=("dpd_30_plus_count", "mean"),
            payment_history_max_dpd=("max_numeric_dpd", "max"),
            payment_history_mean_dpd=("mean_numeric_dpd", "mean"),
            accounts_with_numeric_30_plus=("has_numeric_30_plus", "sum")
        )
        .reset_index()
    )

    # Non-numeric delinquency/status codes are counted separately.
    combined_text = (
        payment_work[payment_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.upper()
    )
    payment_work["severe_status_count"] = combined_text.str.count(
        r"SUB|DBT|LSS"
    )
    severe_features = (
        payment_work.groupby("customer_no")["severe_status_count"]
        .sum()
        .rename("severe_payment_status_count")
        .reset_index()
    )
    payment_features = payment_features.merge(
        severe_features, on="customer_no", how="left"
    )
else:
    payment_features = pd.DataFrame({"customer_no": acct_work["customer_no"].drop_duplicates()})

# ---------- Enquiry features ----------
enq_work["enq_amt"] = pd.to_numeric(enq_work["enq_amt"], errors="coerce")
for col in ["enquiry_dt", "dt_opened", "upload_dt"]:
    if col in enq_work.columns:
        enq_work[col] = pd.to_datetime(enq_work[col], errors="coerce")

reference_date = enq_work["enquiry_dt"].max()

enquiry_features = (
    enq_work.groupby("customer_no")
    .agg(
        enquiry_count=("customer_no", "size"),
        total_enquiry_amount=("enq_amt", "sum"),
        mean_enquiry_amount=("enq_amt", "mean"),
        first_enquiry_date=("enquiry_dt", "min"),
        last_enquiry_date=("enquiry_dt", "max")
    )
    .reset_index()
)

enquiry_features["days_since_last_enquiry"] = (
    reference_date - enquiry_features["last_enquiry_date"]
).dt.days

enq_work["recent_90"] = enq_work["enquiry_dt"].between(
    reference_date - pd.Timedelta(days=90), reference_date
).astype(int)
enq_work["recent_365"] = enq_work["enquiry_dt"].between(
    reference_date - pd.Timedelta(days=365), reference_date
).astype(int)

recency = (
    enq_work.groupby("customer_no")
    .agg(
        count_enquiry_recency_90=("recent_90", "sum"),
        count_enquiry_recency_365=("recent_365", "sum")
    )
    .reset_index()
)

enquiry_features = enquiry_features.merge(
    recency, on="customer_no", how="left"
)

if "dt_opened" in enq_work.columns:
    enq_work["diff_open_enquiry"] = (
        enq_work["enquiry_dt"] - enq_work["dt_opened"]
    ).dt.days

    open_enq = (
        enq_work.groupby("customer_no")["diff_open_enquiry"]
        .mean()
        .rename("mean_diff_open_enquiry_dt")
        .reset_index()
    )
    enquiry_features = enquiry_features.merge(
        open_enq, on="customer_no", how="left"
    )

# Enquiry-purpose diversity/concentration
if "enq_purpose" in enq_work.columns:
    purpose_nunique = (
        enq_work.groupby("customer_no")["enq_purpose"]
        .nunique(dropna=True)
        .rename("unique_enquiry_purposes")
        .reset_index()
    )
    enquiry_features = enquiry_features.merge(
        purpose_nunique, on="customer_no", how="left"
    )

# ---------- Full customer-level matrix ----------
model_full = demo_work.copy()

merge_groups = [
    account_base,
    payment_features,
    enquiry_features
]

for features in merge_groups:
    if not features.empty:
        model_full = model_full.merge(
            features,
            on="customer_no",
            how="left"
        )

# Remove date fields from the predictor matrix but retain them for reference.
date_like = [
    c for c in model_full.columns
    if c in ["dt_opened", "entry_time"]
    or c.endswith("_date")
]

# Ensure target is numeric.
model_full[TARGET] = pd.to_numeric(
    model_full[TARGET], errors="coerce"
)

model_full = model_full.loc[
    model_full[TARGET].isin([0, 1])
].copy()

# Candidate columns:
# all anonymized feature_1...feature_79 + engineered numeric/categorical features.
exclude = {"customer_no", TARGET}
exclude.update(date_like)

candidate_features_full = [
    c for c in model_full.columns
    if c not in exclude
]

# Drop constants only.
constant_cols_full = [
    c for c in candidate_features_full
    if model_full[c].nunique(dropna=False) <= 1
]

candidate_features_full = [
    c for c in candidate_features_full
    if c not in constant_cols_full
]

print("Full modelling matrix:", model_full.shape)
print("Number of candidate predictors:", len(candidate_features_full))
print("Target distribution:")
display(model_full[TARGET].value_counts(normalize=True).rename("proportion"))

# Step 37 — Robust Preprocessing and Baseline Models

### Definition
Split the customer-level data into training, validation and final test populations and build preprocessing pipelines for numerical and categorical predictors.

### Why this step is important
Preprocessing statistics must not be learned from the final test population. Baseline models provide a reference for later nonlinear models.

### Expected Output
Training/validation/test sets plus baseline ROC-AUC and Gini results.

### Insight
The validation population is used for model development and tuning while the final test population remains untouched until Step 40.

In [ ]:
# Step 37 — Train/validation/test split

X_all = model_full[candidate_features_full].copy()
y_all = model_full[TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_all
)

X_tr, X_valid, y_tr, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

numeric_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [
    c for c in X_tr.columns
    if c not in numeric_cols
]

print("Training:", X_tr.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)
print("Numeric predictors:", len(numeric_cols))
print("Categorical predictors:", len(categorical_cols))

In [ ]:
# Preprocessing for linear/tree models

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=5
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ],
    remainder="drop"
)

def gini_score(y_true, probability):
    return 2 * roc_auc_score(y_true, probability) - 1

logistic_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(
        max_iter=3000,
        C=0.5,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_tr, y_tr)
log_valid_prob = logistic_model.predict_proba(X_valid)[:, 1]

print("Logistic validation AUC :", round(roc_auc_score(y_valid, log_valid_prob), 4))
print("Logistic validation Gini:", round(gini_score(y_valid, log_valid_prob), 4))

In [ ]:
# ExtraTrees baseline

extra_model = Pipeline([
    ("prep", preprocessor),
    ("model", ExtraTreesClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

extra_model.fit(X_tr, y_tr)
extra_valid_prob = extra_model.predict_proba(X_valid)[:, 1]

print("ExtraTrees validation AUC :", round(roc_auc_score(y_valid, extra_valid_prob), 4))
print("ExtraTrees validation Gini:", round(gini_score(y_valid, extra_valid_prob), 4))

# Step 38 — Gradient Boosting / XGBoost Candidate

### Definition
Train a boosted-tree model capable of learning nonlinear relationships and interactions in the credit-risk predictors.

### Why this step is important
Credit-risk relationships may not be linear. Boosting can capture interactions between payment history, utilization, account behaviour, enquiries and application features.

### Expected Output
A boosted-tree candidate and validation probability scores with ROC-AUC/Gini.

### Insight
The candidate is evaluated with ROC-AUC and converted to Gini using `Gini = 2 × ROC-AUC − 1`.

In [ ]:
# Step 38 — XGBoost candidate

xgb_model = None
xgb_valid_prob = None

try:
    from xgboost import XGBClassifier

    xgb_model = Pipeline([
        ("prep", preprocessor),
        ("model", XGBClassifier(
            n_estimators=700,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=5,
            subsample=0.85,
            colsample_bytree=0.85,
            gamma=0,
            reg_alpha=0.05,
            reg_lambda=3.0,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    xgb_model.fit(X_tr, y_tr)
    xgb_valid_prob = xgb_model.predict_proba(X_valid)[:, 1]

    print("XGBoost validation AUC :",
          round(roc_auc_score(y_valid, xgb_valid_prob), 4))
    print("XGBoost validation Gini:",
          round(gini_score(y_valid, xgb_valid_prob), 4))

except ImportError:
    print("XGBoost is not installed. The notebook will use ExtraTrees as the main tree model.")

# Step 39 — Tune the Strongest Model and Build an Ensemble

### Definition
Tune the strongest nonlinear candidate using cross-validated ROC-AUC and, where useful, combine validated probability scores from multiple models.

### Why this step is important
Systematic tuning can improve discrimination without using the final test population. A validated probability blend can combine complementary model behaviour.

### Expected Output
Best parameters, validation ROC-AUC/Gini, and validated blend performance.

### Insight
All tuning and blend-weight decisions are made using training/validation data. The final test population remains untouched until Step 40.

In [ ]:
# Step 39 — Tune XGBoost if available; otherwise tune ExtraTrees

if xgb_model is not None:
    tuning_estimator = xgb_model

    param_distributions = {
        "model__n_estimators": [400, 600, 800, 1000],
        "model__learning_rate": [0.015, 0.025, 0.035, 0.05],
        "model__max_depth": [2, 3, 4, 5],
        "model__min_child_weight": [3, 5, 10, 20],
        "model__subsample": [0.70, 0.80, 0.90, 1.0],
        "model__colsample_bytree": [0.70, 0.80, 0.90, 1.0],
        "model__gamma": [0, 0.05, 0.10, 0.25],
        "model__reg_alpha": [0, 0.05, 0.10, 0.25],
        "model__reg_lambda": [1, 2, 3, 5, 8]
    }
else:
    tuning_estimator = extra_model

    param_distributions = {
        "model__n_estimators": [500, 700, 1000],
        "model__min_samples_leaf": [5, 10, 20, 30],
        "model__max_features": ["sqrt", "log2", 0.5, 0.8],
        "model__max_depth": [None, 10, 20, 30]
    }

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

search = RandomizedSearchCV(
    estimator=tuning_estimator,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True
)

search.fit(X_tr, y_tr)

tuned_valid_prob = search.predict_proba(X_valid)[:, 1]
tuned_valid_gini = gini_score(y_valid, tuned_valid_prob)

print("Tuned validation AUC :",
      round(roc_auc_score(y_valid, tuned_valid_prob), 4))
print("Tuned validation Gini:",
      round(tuned_valid_gini, 4))
print("\nBest parameters:")
print(search.best_params_)

In [ ]:
# Probability blend:
# Blend only models that actually improve the validation ranking.

blend_candidates = {
    "tuned": tuned_valid_prob,
    "logistic": log_valid_prob,
    "extra_trees": extra_valid_prob
}

if xgb_valid_prob is not None:
    blend_candidates["xgb_base"] = xgb_valid_prob

blend_results = []

# Grid over the weight assigned to the tuned model and the logistic model;
# the remaining weight goes to ExtraTrees.
for tuned_w in np.arange(0.50, 1.01, 0.05):
    for log_w in np.arange(0.00, min(0.51, 1.01 - tuned_w), 0.05):
        extra_w = 1 - tuned_w - log_w
        if extra_w < 0:
            continue

        blend_prob = (
            tuned_w * tuned_valid_prob
            + log_w * log_valid_prob
            + extra_w * extra_valid_prob
        )

        blend_auc = roc_auc_score(y_valid, blend_prob)

        blend_results.append({
            "tuned_weight": tuned_w,
            "logistic_weight": log_w,
            "extra_trees_weight": extra_w,
            "AUC": blend_auc,
            "Gini": 2 * blend_auc - 1
        })

blend_results = pd.DataFrame(blend_results).sort_values(
    "Gini", ascending=False
)

display(blend_results.head(10))

best_blend = blend_results.iloc[0]

print("Best validation blend Gini:",
      round(best_blend["Gini"], 4))
print("Benchmark Gini:",
      BENCHMARK_GINI)

# Step 40 — Final Test Gini and Rank Ordering

### Definition
Refit the selected model components on the complete training population, evaluate once on the held-out test population, and rank customers into ten risk deciles by predicted bad-credit probability.

### Why this step is important
The project requires model evaluation using both Gini and rank ordering.

### Benchmark
**Gini = 0.379 (37.9%)**

### Expected Output
Final test ROC-AUC, final test Gini, benchmark status, risk-decile table and rank-ordering chart.

### Insight
Benchmark achievement is determined only from the actual held-out test Gini. The decile analysis shows whether observed bad rates are ordered consistently with predicted risk.

In [ ]:
# Step 40 — Refit models on the complete training population

# Refit tuned model
final_tuned_model = search.best_estimator_
final_tuned_model.fit(X_train, y_train)

final_tuned_test_prob = final_tuned_model.predict_proba(X_test)[:, 1]

# Refit logistic and ExtraTrees
logistic_model.fit(X_train, y_train)
extra_model.fit(X_train, y_train)

final_log_test_prob = logistic_model.predict_proba(X_test)[:, 1]
final_extra_test_prob = extra_model.predict_proba(X_test)[:, 1]

# Apply validation-selected blend weights
tuned_w = float(best_blend["tuned_weight"])
log_w = float(best_blend["logistic_weight"])
extra_w = float(best_blend["extra_trees_weight"])

final_test_prob = (
    tuned_w * final_tuned_test_prob
    + log_w * final_log_test_prob
    + extra_w * final_extra_test_prob
)

final_auc = roc_auc_score(y_test, final_test_prob)
final_gini = 2 * final_auc - 1

print(f"Final test ROC-AUC : {final_auc:.4f}")
print(f"Final test Gini    : {final_gini:.4f}")
print(f"Benchmark Gini     : {BENCHMARK_GINI:.4f}")
print(f"Gini gap           : {BENCHMARK_GINI - final_gini:.4f}")

if final_gini >= BENCHMARK_GINI:
    print("\nBenchmark status: ACHIEVED")
else:
    print("\nBenchmark status: NOT YET ACHIEVED")

In [ ]:
# Step 40B — Final risk rank ordering

rank_df = pd.DataFrame({
    "Bad_Probability": final_test_prob,
    "Bad_label": y_test.to_numpy()
})

rank_df = rank_df.sort_values(
    "Bad_Probability",
    ascending=False
).reset_index(drop=True)

rank_df["Decile"] = pd.qcut(
    rank_df.index,
    q=10,
    labels=False,
    duplicates="drop"
) + 1

decile_summary = (
    rank_df.groupby("Decile")
    .agg(
        Customers=("Bad_label", "size"),
        Bad_Customers=("Bad_label", "sum"),
        Mean_Predicted_Risk=("Bad_Probability", "mean"),
        Bad_Rate=("Bad_label", "mean")
    )
    .reset_index()
)

decile_summary["Bad_Rate"] *= 100

display(decile_summary)

plt.figure(figsize=(10, 6))
plt.plot(
    decile_summary["Decile"],
    decile_summary["Bad_Rate"],
    marker="o"
)
plt.xlabel("Risk Decile (1 = Highest Predicted Risk)")
plt.ylabel("Observed Bad Rate (%)")
plt.title("GoodCredit Final Model — Risk Rank Ordering")
plt.xticks(decile_summary["Decile"])
plt.grid(alpha=0.3)
plt.show()

# Step 41 — Final Model Performance Summary

**Definition:** Consolidate the final held-out model performance.

**Why it matters:** Provides one clear benchmark result for the submission.

**Expected output:** ROC-AUC, Gini, benchmark, gap, and benchmark status.

**Insight:** The status is calculated from the actual test result.

### Insight
This output provides the evidence used in the final Bank GoodCredit project evaluation.

In [ ]:
# Step 41 — Final performance summary
BENCHMARK_GINI = 0.379

if "test_prob" not in globals():
    if "final_test_prob" in globals():
        test_prob = final_test_prob
    else:
        raise NameError("Run Step 40 first.")

test_auc = roc_auc_score(y_test, test_prob)
test_gini = 2 * test_auc - 1

performance_summary = pd.DataFrame({
    "Metric": ["Test ROC-AUC", "Test Gini", "Benchmark Gini", "Gini Gap"],
    "Value": [test_auc, test_gini, BENCHMARK_GINI, test_gini - BENCHMARK_GINI]
})
display(performance_summary)

print("Benchmark status:",
      "ACHIEVED" if test_gini >= BENCHMARK_GINI else "NOT YET ACHIEVED")

# Step 42 — Final Feature Gain / Importance

**Definition:** Rank predictors by their contribution to the final model.

**Why it matters:** The project specifically requires selected features with gain.

**Expected output:** Ranked feature-importance table and chart.

**Insight:** Importance indicates model contribution, not causality.

### Insight
This output provides the evidence used in the final Bank GoodCredit project evaluation.

In [ ]:
# Step 42 — Feature importance
feature_names = list(X_train.columns)

model_candidates = [
    globals().get("final_model"),
    globals().get("final_tuned_model"),
    globals().get("search_best_model")
]
final_model_for_importance = next((m for m in model_candidates if m is not None), None)

if final_model_for_importance is None and "search" in globals():
    final_model_for_importance = search.best_estimator_

if final_model_for_importance is None:
    raise NameError("Run Step 40 first.")

underlying = final_model_for_importance
if hasattr(final_model_for_importance, "named_steps"):
    underlying = list(final_model_for_importance.named_steps.values())[-1]

importance = None
method = None

if hasattr(underlying, "feature_importances_"):
    importance = np.asarray(underlying.feature_importances_)
    method = "Native feature importance"
elif hasattr(underlying, "coef_"):
    importance = np.abs(np.asarray(underlying.coef_).ravel())
    method = "Absolute model coefficient"

if importance is not None and len(importance) == len(feature_names):
    feature_gain = pd.DataFrame({
        "Feature": feature_names,
        "Gain_or_Importance": importance
    }).sort_values("Gain_or_Importance", ascending=False).reset_index(drop=True)
    feature_gain["Rank"] = np.arange(1, len(feature_gain) + 1)
    total = feature_gain["Gain_or_Importance"].sum()
    feature_gain["Relative_Percentage"] = (
        feature_gain["Gain_or_Importance"] / total * 100 if total > 0 else 0
    )
else:
    from sklearn.inspection import permutation_importance
    n = min(3000, len(X_test))
    idx = X_test.sample(n=n, random_state=42).index
    perm = permutation_importance(
        final_model_for_importance, X_test.loc[idx], y_test.loc[idx],
        scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1
    )
    feature_gain = pd.DataFrame({
        "Feature": feature_names,
        "Gain_or_Importance": perm.importances_mean
    }).sort_values("Gain_or_Importance", ascending=False).reset_index(drop=True)
    feature_gain["Rank"] = np.arange(1, len(feature_gain) + 1)
    pos = feature_gain["Gain_or_Importance"].clip(lower=0)
    feature_gain["Relative_Percentage"] = pos / pos.sum() * 100 if pos.sum() > 0 else 0
    method = "Permutation importance"

print("Importance method:", method)
display(feature_gain.head(20))

In [ ]:
# Top feature visualization
top = feature_gain.head(15).sort_values("Gain_or_Importance")
plt.figure(figsize=(10, 7))
plt.barh(top["Feature"], top["Gain_or_Importance"])
plt.xlabel("Gain / Importance")
plt.ylabel("Feature")
plt.title("Top Final Model Features")
plt.tight_layout()
plt.show()

# Step 43 — Final Risk Rank Ordering

**Definition:** Sort customers by predicted bad-credit probability and divide them into ten deciles.

**Why it matters:** Rank ordering is a required project evaluation.

**Expected output:** Decile table and bad-rate chart.

**Insight:** Observed bad rates should be examined across increasing predicted-risk groups.

### Insight
This output provides the evidence used in the final Bank GoodCredit project evaluation.

In [ ]:
# Step 43 — Final decile rank ordering
rank_df = pd.DataFrame({
    "Predicted_Bad_Probability": np.asarray(test_prob),
    "Bad_label": np.asarray(y_test)
}).sort_values("Predicted_Bad_Probability", ascending=False).reset_index(drop=True)

rank_df["Decile"] = (
    pd.qcut(np.arange(len(rank_df)), q=10, labels=False, duplicates="drop") + 1
)

rank_ordering = rank_df.groupby("Decile").agg(
    Customers=("Bad_label", "size"),
    Bad_Customers=("Bad_label", "sum"),
    Mean_Predicted_Bad_Probability=("Predicted_Bad_Probability", "mean"),
    Bad_Rate=("Bad_label", "mean")
).reset_index()

rank_ordering["Bad_Rate_Percent"] = rank_ordering["Bad_Rate"] * 100
rank_ordering["Population_Percent"] = (
    rank_ordering["Customers"] / rank_ordering["Customers"].sum() * 100
)
display(rank_ordering)

In [ ]:
# Rank-ordering chart
plt.figure(figsize=(10, 6))
plt.plot(rank_ordering["Decile"], rank_ordering["Bad_Rate_Percent"], marker="o")
plt.xlabel("Risk Decile (1 = Highest Predicted Risk)")
plt.ylabel("Observed Bad Rate (%)")
plt.title("Final Bank GoodCredit Rank Ordering")
plt.xticks(rank_ordering["Decile"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Step 44 — Business Insights & Credit-Risk Decisions

### Definition
Translate exploratory findings and final model evidence into business-facing observations and decisions.

### Why this step is important
The project explicitly asks what was found and what decision was taken.

### Evidence Areas
Payment-history behaviour, account duration, credit utilization, recent enquiries and enquiry-purpose features are central modelling themes in the project documentation.

### Expected Output
A concise **Observation → Evidence → Business implication → Decision** table.

### Insight
Business conclusions must be grounded in the actual feature-gain and rank-ordering outputs. Anonymized `feature_1`–`feature_79` variables should not be assigned unsupported business meanings.

In [ ]:
# Step 44 — Display the model's strongest evidence for business discussion
business_evidence = feature_gain.head(15)[
    ["Rank", "Feature", "Gain_or_Importance", "Relative_Percentage"]
].copy()

display(business_evidence)

print("Use the ranked features above to support the final business discussion.")
print("Do not assign a business meaning to anonymized feature_* variables unless it is documented.")

# Step 45 — Final Project Conclusion

### Definition
Present the final Bank GoodCredit project conclusion in a clear, submission-ready format.

### Why this step is important
This closes the workflow by connecting the business objective, modelling methodology, benchmark performance, feature evidence and rank ordering.

### Final Conclusion
The project builds a customer-level credit-risk model using Account/payment history, Enquiry history and current customer demographic/application information. The target is `Bad_label`, where `0` represents good credit history and `1` represents bad credit history associated with the 30+ DPD bucket.

The final model is evaluated against the project benchmark of **Gini = 0.379 (37.9%)** using a held-out test population. The executable summary below reports the actual Gini, benchmark status, top features and final risk-ordering evidence.

### Expected Output
Final ROC-AUC, Gini, benchmark status, top model features, rank ordering, business implications and limitations.

### Insight
The project is submission-ready when these outputs have been generated successfully from a clean end-to-end notebook run.

In [ ]:
# Step 45 — Final submission summary
status = "ACHIEVED" if test_gini >= BENCHMARK_GINI else "NOT YET ACHIEVED"

print("=" * 70)
print("BANK GOODCREDIT — FINAL PROJECT SUMMARY")
print("=" * 70)
print(f"Test ROC-AUC : {test_auc:.4f}")
print(f"Test Gini    : {test_gini:.4f}")
print(f"Benchmark    : {BENCHMARK_GINI:.4f}")
print(f"Status       : {status}")

print("\nTop 10 model features:")
for _, row in feature_gain.head(10).iterrows():
    print(f"{int(row['Rank']):2d}. {row['Feature']}")

print("\nRisk rank ordering:")
print("Customers were ranked by predicted bad-credit probability and divided into deciles.")

print("\nBusiness interpretation:")
print("Payment history, account behaviour, credit utilization, enquiry activity and")
print("other customer-level predictors were incorporated into the credit-risk model.")

print("\nLimitations:")
print("Anonymized demographic variables cannot be assigned business meaning beyond")
print("their measured predictive contribution. Future populations should be monitored.")

print("\nFinal decision:")
print(
    "The held-out test Gini meets or exceeds the project benchmark."
    if status == "ACHIEVED"
    else
    "The held-out test Gini is below the project benchmark; further modelling is required."
)

# Final Requirement Audit — Bank GoodCredit PR-0015

The source project defines three core deliverables: **data-exploration insights and decisions; a feature matrix with selected features and gain; and model evaluation using Gini and rank ordering.**

| Requirement | Coverage |
|---|---|
| Business problem and target | Steps 1, 9 |
| Three source tables | Step 4 |
| Data quality | Steps 6–8, 21–23 |
| Customer-level relationships | Steps 9, 15, 20 |
| Payment-history features | Steps 14, 26, 36 |
| Account-duration features | Steps 18, 27, 36 |
| Utilization features | Steps 28, 36 |
| Enquiry-recency features | Steps 19, 29, 36 |
| Leakage screening | Step 31 |
| Categorical treatment | Steps 32, 37 |
| Correlation analysis | Step 33 |
| Missing-value treatment | Steps 34, 37 |
| Train/test methodology | Steps 35, 37 |
| Model development | Steps 37–39 |
| Gini evaluation | Steps 40–41 |
| Benchmark comparison | Steps 40–41 |
| Feature gain/importance | Step 42 |
| Rank ordering | Steps 40, 43 |
| Business insights/decisions | Step 44 |
| Final conclusion | Step 45 |

### Important distinction
The project document provides **reference feature suggestions and reference gains**. Step 42 calculates **actual model-derived importance/gain**. These should be reported separately rather than treated as identical measures.

### Benchmark
**Gini = 0.379 (37.9%)**.

# Final End-to-End Workflow Diagram

This diagram summarizes the complete Bank GoodCredit workflow from source data through the final business output.

In [ ]:
# Bank GoodCredit — End-to-End Credit Risk Modelling Workflow

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

workflow = [
    ("SOURCE DATA", "Account • Enquiry • Demographics"),
    ("DATA PREPARATION", "Quality checks • Missing values • Dates"),
    ("FEATURE ENGINEERING", "Payment history • Duration • Utilization • Enquiries"),
    ("CUSTOMER FEATURE MATRIX", "One row per customer • Bad_label target"),
    ("MODEL PREPARATION", "Leakage check • Encoding • Imputation • Train / Test"),
    ("MODEL BUILDING", "Logistic Regression • Tree Models • Boosting"),
    ("MODEL EVALUATION", "ROC-AUC • Gini • Benchmark = 0.379"),
    ("RISK RANK ORDERING", "Risk deciles • Bad-rate analysis"),
    ("BUSINESS OUTPUT", "Feature gain • Insights • Final conclusion")
]

fig, ax = plt.subplots(figsize=(12, 15))
ax.set_xlim(0, 10)
ax.set_ylim(0, 15)
ax.axis("off")

ax.set_title(
    "Bank GoodCredit — End-to-End Credit Risk Modelling Workflow",
    fontsize=16,
    fontweight="bold",
    pad=20
)

box_x = 1.5
box_width = 7
box_height = 0.85
start_y = 13.2
gap = 1.45

positions = []

for i, (title, description) in enumerate(workflow):
    y = start_y - i * gap
    positions.append(y)

    box = FancyBboxPatch(
        (box_x, y),
        box_width,
        box_height,
        boxstyle="round,pad=0.04",
        linewidth=1.5
    )
    ax.add_patch(box)

    ax.text(
        5, y + 0.55, title,
        ha="center", va="center",
        fontsize=11, fontweight="bold"
    )

    ax.text(
        5, y + 0.25, description,
        ha="center", va="center",
        fontsize=9
    )

for i in range(len(positions) - 1):
    ax.annotate(
        "",
        xy=(5, positions[i + 1] + box_height + 0.05),
        xytext=(5, positions[i] - 0.05),
        arrowprops=dict(arrowstyle="->", linewidth=1.5)
    )

plt.tight_layout()
plt.show()


# Final Project Summary

This project developed a **credit-risk prediction model for Bank GoodCredit** to identify current credit-card customers with a higher risk of bad credit history. The target variable, **`Bad_label`**, represents customer credit history, where `0` indicates good credit history and `1` indicates bad credit history associated with the 30+ DPD bucket.

The project used three primary data sources: **Customer Account Data, Customer Enquiry Data, and Customer Demographics/Application Data**. The data was cleaned, validated, transformed and aggregated into a customer-level feature matrix.

Feature engineering incorporated **payment-history behaviour, account duration, credit utilization, enquiry recency and enquiry behaviour**, together with the anonymized customer/application features supplied in the project data.

Multiple classification approaches were evaluated, including **Logistic Regression, ExtraTrees and XGBoost/boosting-based modelling**. Model selection and tuning were performed using training and validation data, while the final model was evaluated on a held-out test population.

Model performance was evaluated using **ROC-AUC and Gini**, with **risk-decile rank ordering** used to assess how well predicted risk separates customers into different observed bad-rate groups.

The project benchmark is **Gini = 0.379 (37.9%)**. The actual final test Gini and benchmark status are reported in the final performance output above.

The final analysis also provides a **feature-gain/importance table**, a **risk-decile rank-ordering table and chart**, and business-oriented interpretation of the strongest observed predictors.

## Final Project Deliverables

- Data exploration and business insights
- Customer-level feature engineering
- Final feature matrix with model importance/gain
- Classification model development and tuning
- **Gini benchmark evaluation**
- Risk-decile rank ordering
- Business interpretation and credit-risk decisions
- Final model conclusion

## Conclusion

Overall, the Bank GoodCredit project demonstrates how customer account history, payment behaviour, credit utilization, enquiry activity and application-level information can be combined to build a reproducible **credit-risk prediction and customer risk-ranking framework**.

The final notebook provides the required evidence for the project: **data-exploration insights and decisions, a feature matrix with selectedfeatures and gain, and model evaluation using Gini and rank ordering**.
